# Model Optimization
## Objective
The objective of this notebook is to improve the performance and reliability of the Random Forest regression model.

This notebook evaluates the model using cross-validation and optimizes its hyperparameters using RandomizedSearchCV.

The optimized model will be used for deployment in the application.

In [2]:
import pandas as pd
import numpy as np

import joblib

from sklearn.model_selection import (
    train_test_split,
    RandomizedSearchCV,
    cross_val_score
)

from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

import plotly.express as px

In [3]:
df = pd.read_csv("../data/processed/uber_feature_engineered.csv")

In [4]:
drop_columns = [
    "key",
    "pickup_datetime",
    "pickup_day"
]

X = df.drop(
    columns=drop_columns + ["fare_amount"]
)

y = df["fare_amount"]

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

## Baseline Model

Before tuning, a baseline Random Forest model is trained.
This provides a reference point for evaluating the impact of hyperparameter optimization.

In [6]:
baseline_rf = RandomForestRegressor(
    random_state=42,
    n_estimators=100,
    n_jobs=-1
)

baseline_rf.fit(
    X_train,
    y_train
)

baseline_pred = baseline_rf.predict(X_test)

In [7]:
def evaluate(y_true, y_pred):

    mae = mean_absolute_error(y_true, y_pred)

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    r2 = r2_score(
        y_true,
        y_pred
    )

    return mae, rmse, r2

In [8]:
baseline_mae, baseline_rmse, baseline_r2 = evaluate(
    y_test,
    baseline_pred
)

## Cross Validation

A single train-test split may not provide a complete picture of model performance.
Five-fold cross-validation evaluates the model on multiple subsets of the data, providing a more robust estimate of generalization performance.

In [9]:
cv_scores = cross_val_score(
    baseline_rf,
    X,
    y,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

In [10]:
print("Cross Validation Scores")

print(cv_scores)

print()

print("Average R²")

print(cv_scores.mean())

Cross Validation Scores
[0.82248948 0.84732853 0.84184489 0.81433435 0.78750724]

Average R²
0.8227008968673288


In [11]:
cv_df = pd.DataFrame({
    "Fold": range(1,6),
    "R² Score": cv_scores
})

fig = px.bar(
    cv_df,
    x="Fold",
    y="R² Score",
    text="R² Score",
    color="R² Score",
    title="5-Fold Cross Validation Scores",
    template="plotly_white"
)

fig.update_layout(
    title_x=0.5
)

fig.show()

### Observation
The cross-validation scores are consistent across all folds.

### Business Insight
The model demonstrates stable generalization performance and is unlikely to be overfitting to a single train-test split.

In [20]:
param_distributions = {
    "n_estimators": [100, 200, 300, 500],
    "max_depth": [10, 20, 30, 40, None],
    "min_samples_split": [2, 5, 10, 15],
    "min_samples_leaf": [1, 2, 4, 8],
    "max_features": ["sqrt", "log2"],
    "bootstrap": [True, False]
}

In [13]:
random_search = RandomizedSearchCV(

    estimator=RandomForestRegressor(
        random_state=42
    ),

    param_distributions=param_grid,

    n_iter=20,

    cv=5,

    scoring="r2",

    random_state=42,

    verbose=2,

    n_jobs=-1
)

In [14]:
random_search.fit(
    X_train,
    y_train
)

Fitting 5 folds for each of 20 candidates, totalling 100 fits


c:\Users\vgaik\OneDrive\Desktop\Uber Fare prediction and Analytics platform\venv\Lib\site-packages\sklearn\model_selection\_validation.py:489: FitFailedWarning: 
8 fits failed out of a total of 100.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
5 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\vgaik\OneDrive\Desktop\Uber Fare prediction and Analytics platform\venv\Lib\site-packages\sklearn\model_selection\_validation.py", line 851, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\vgaik\OneDrive\Desktop\Uber Fare prediction and Analytics platform\venv\Lib\site-packages\sklearn\base.py", line 1403, in wrapper
  

,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",RandomForestR...ndom_state=42)
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'max_depth': [10, 20, ...], 'max_features': ['sqrt', 'log2'], 'min_samples_leaf': [1, 2, ...], 'min_samples_split': [2, 5, ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",20
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",5
,"verbose verbose: int, default = 0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",2
,"random_state random_state: int, RandomState instance or None, default=NonePseudo random number generator state used for random uniform samplingfrom lists of possible values instead of scipy.stats distributions.Pass an int for reproducible output across multiplefunction calls.See :term:`Glossary <random_state>`.",42
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichretur

In [15]:
print(random_search.best_params_)

{'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': 30}


In [16]:
print(random_search.best_score_)

0.8250852615107409


In [17]:
best_model = random_search.best_estimator_

In [18]:
optimized_pred = best_model.predict(X_test)

In [19]:
optimized_mae, optimized_rmse, optimized_r2 = evaluate(
    y_test,
    optimized_pred
)

print(optimized_mae)
print(optimized_rmse)
print(optimized_r2)

1.7779805640530537
3.7082206657187995
0.8523982900832763


In [22]:
random_search.cv_results_

{'mean_fit_time': array([ 66.43934741, 121.77305946,  57.86252141,  89.17974324,
        123.01393862,  64.46616077,  86.92206001,  48.34948812,
         90.2743876 , 189.2679276 , 101.58811851,  58.30072823,
         47.01917081, 258.45141263,  28.16605797,  61.79002776,
         62.78038278,  50.51126018,  97.10216327, 184.22533073]),
 'std_fit_time': array([ 5.55454918,  6.36406158,  3.02404981,  3.65092095,  2.56758721,
        10.31400984,  0.89319358,  0.84586621,  1.40277109, 56.83544677,
        58.37910589,  0.90495734,  1.77986315,  6.81515524,  0.91434171,
         5.73300054,  3.92413406,  1.39438381,  2.24325946, 11.65772141]),
 'mean_score_time': array([ 0.86492414,  6.86800151,  2.64154482,  1.41808906, 33.0551013 ,
         3.06857004,  1.18615017,  1.15055056,  2.08241611,  3.6212564 ,
         2.11272745,  0.76622992,  0.88164349, 11.35380049,  0.35988832,
         3.50941629,  0.75325561,  1.17778411,  3.11077695,  7.62669425]),
 'std_score_time': array([0.08848155, 

In [23]:
search_results = pd.DataFrame(random_search.cv_results_)

search_results.to_csv(
    "../reports/random_search_results.csv",
    index=False
)

In [26]:
comparison = pd.DataFrame({
    "Model": [
        "Baseline Random Forest",
        "Optimized Random Forest"
    ],
    "MAE": [
        baseline_mae,
        optimized_mae
    ],
    "RMSE": [
        baseline_rmse,
        optimized_rmse
    ],
    "R²": [
        baseline_r2,
        optimized_r2
    ]
})

comparison

,Model,MAE,RMSE,R²
0,Baseline Random Forest,1.813140,3.684960,0.854244
1,Optimized Random Forest,1.777981,3.708221,0.852398


In [27]:
import plotly.express as px

fig = px.bar(
    comparison,
    x="Model",
    y="R²",
    color="R²",
    text="R²",
    title="Baseline vs Optimized Random Forest",
    template="plotly_white"
)

fig.update_layout(
    title_x=0.5,
    height=500
)

fig.show()

In [28]:
comparison.to_csv(
    "../reports/model_comparison.csv",
    index=False
)